# 05 — Regime-Specific LSTM Training (Phase 3b)

This notebook runs `src/train_LSTM_regime.py`, which executes the full Phase 3b pipeline:

1. Load the train / val / test splits from `data/processed/`.
2. Merge Viterbi regime labels from `data/processed/regime_probabilities.parquet`.
3. For each regime (`calm`, `volatile`):
   - Build a **`RegimeWindowDataset`** that retains only windows whose dominant Viterbi state matches the regime.
   - Run an **Optuna** (TPE) hyperparameter search scored on regime-filtered val MSE.
   - Retrain with the best params using early-stopping.
   - Evaluate on the **full** test set (both regimes) — required for ensemble blending in Phase 4.
   - Save `models/lstm_{regime}.pt` and `models/lstm_{regime}_scaler.joblib`.

**Key design decisions:**
- A window spanning both regimes is assigned to its *majority* (dominant) regime.
- The feature scaler is always fit on the **full** training split, not the regime-filtered subset, to keep the input scale consistent across all three models.
- The val DataLoader is also regime-filtered so the calm LSTM is not penalised for poor volatile-regime predictions during tuning.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_regime.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
print(f"Script: {SCRIPT}")
print(f"Python: {sys.executable}")

## Run the regime training pipeline

Tweak `ARGS` below to control the trial count, epochs, or feature list. `--regime both` trains calm and volatile sequentially.

In [ ]:
ARGS = [
    "--regime", "both",
    # "--n-trials", "30",
    # "--tune-epochs", "30",
    # "--final-epochs", "80",
]

result = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)] + ARGS,
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)
combined_output = result.stdout + result.stderr
print(combined_output[-6000:])  # last 6000 chars
if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code {result.returncode}")

## Parse and display the results

In [ ]:
import pandas as pd

marker = "=== Regime-Specific LSTM Results ==="
idx = combined_output.find(marker)
assert idx != -1, "Results marker not found in script output."
json_blob = combined_output[idx + len(marker):].strip()
results = json.loads(json_blob)

rows = []
for regime, r in results.items():
    rows.append({
        "Regime":              regime,
        "Train windows":       r["n_train_windows"],
        "Val windows":         r["n_val_windows"],
        "Best val MSE (raw)": r["best_val_mse_raw"],
        "Test MSE":            r["test_metrics"]["MSE"],
        "Test RMSE":           r["test_metrics"]["RMSE"],
        "Test MAE":            r["test_metrics"]["MAE"],
        "hidden_size":         r["best_params"]["hidden_size"],
        "n_layers":            r["best_params"]["n_layers"],
        "seq_len":             r["best_params"]["seq_len"],
        "lr":                  f"{r['best_params']['lr']:.2e}",
    })

df = pd.DataFrame(rows).set_index("Regime")
print("=== Regime LSTM comparison ===")
display(df)

## Verify saved artifacts

In [ ]:
import torch, joblib

models_dir = REPO_ROOT / "models"
for regime in ["calm", "volatile"]:
    pt_path     = models_dir / f"lstm_{regime}.pt"
    scaler_path = models_dir / f"lstm_{regime}_scaler.joblib"
    
    assert pt_path.exists(),     f"Missing: {pt_path}"
    assert scaler_path.exists(), f"Missing: {scaler_path}"
    
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
    scaler = joblib.load(scaler_path)
    print(f"[{regime}] Features: {ckpt['features']}")
    print(f"[{regime}] Hyperparams: {ckpt['hyperparameters']}")
    print(f"[{regime}] Scaler mean shape: {scaler.mean_.shape}")
    print()

print("All regime artifacts verified.")

## Saved artifacts

| File | Description |
|------|-------------|
| `models/lstm_calm.pt` | Calm-regime LSTM state_dict + hyperparameters + feature list |
| `models/lstm_calm_scaler.joblib` | Feature StandardScaler (fit on full train) for the calm LSTM |
| `models/lstm_volatile.pt` | Volatile-regime LSTM state_dict + hyperparameters + feature list |
| `models/lstm_volatile_scaler.joblib` | Feature StandardScaler for the volatile LSTM |

These artifacts are inputs to **Phase 4** (`src/ensemble.py`) which blends predictions using HMM soft probabilities.